# 06 — Đánh giá HGB ở cấp từng business

## tl;dr

Báo cáo đọc lại **kết quả đã hoàn tất của notebook 03 và 05**, không huấn luyện lại.
Kết luận bên dưới được tạo từ validation thực tế; không giả định HGB là model thắng.
Đơn vị dự đoán là **Recorded check-ins của một business trong một ngày**.
Ứng dụng có chính sách HGB phục vụ weather scenario riêng; calendar ở đây là baseline/reference.
Chính sách sản phẩm không đồng nghĩa HGB là model thắng accuracy trong backtest.

### Thiết lập và cấu hình

Chạy [03](03_location_forecast_poc.ipynb), rồi [05](05_forecast_gam_catboost_comparison.ipynb)
trước. Dùng môi trường `uv sync --frozen --group notebooks` và nguồn dữ liệu đã import.
Các override bên dưới cho phép chọn bằng chứng cụ thể; để `None` sẽ tìm cặp run COMPLETE
mới nhất còn hợp lệ. Không đọc các artifact ZIP cũ.

In [ ]:
import hashlib
import json
from datetime import UTC, date, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.patches import FancyBboxPatch

from sitesense.business_research import load_models, load_run, recommend

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

# Config: None chọn run COMPLETE mới nhất còn xác minh được code/lockfile/nguồn.
# Có thể đặt đường dẫn tuyệt đối hoặc đường dẫn tính từ ROOT để chọn run cụ thể.
COMPARISON_RUN_OVERRIDE = None
POC_RUN_OVERRIDE = None
EXPECTED_SOURCE_CATEGORY = None  # Ví dụ "Pizza"; None đọc đúng ngách của run 03/05.
DEMO_DAY = date(2026, 7, 15)
TOP_N = 8
COMPARE_BUSINESS_IDS = []  # Rỗng: dùng hai business đầu trong bảng kịch bản.
MANUAL_WEATHER = {"tmin_c": 24.0, "tmax_c": 32.0, "precipitation_mm": 20.0}
WEATHER_EXAMPLES = {
    "Mát, khô · giả định": {"tmin_c": 12.0, "tmax_c": 20.0, "precipitation_mm": 0.0},
    "Nóng, khô · giả định": {"tmin_c": 25.0, "tmax_c": 35.0, "precipitation_mm": 0.0},
    "Tự nhập · giả định": MANUAL_WEATHER,
}
SOURCE = ROOT / "notebooks/06_hgb_model_selection_report.ipynb"
OUTPUT_ROOT = ROOT / "data/processed/business_hgb_selection_report"
PRODUCTION_BUNDLE_DIR = ROOT / "models/business_forecast"

MODEL_NAMES = {
    "overall_mean": "Trung bình riêng mỗi business",
    "month_weekday_shrunk": "Calendar có co về trung bình",
    "poisson": "Poisson regression + weather",
    "hgb": "HGB + weather",
    "gam_poisson": "GAM dạng spline + weather",
    "catboost": "CatBoost + weather",
}
CALENDAR_MODELS = ["overall_mean", "month_weekday_shrunk"]
WEATHER_MODELS = ["poisson", "hgb", "gam_poisson", "catboost"]
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def file_hashes(directory):
    return {
        str(p.relative_to(directory)): sha256(p)
        for p in sorted(directory.rglob("*"))
        if p.is_file()
    }


def configured_path(value):
    path = Path(value)
    return path.resolve() if path.is_absolute() else (ROOT / path).resolve()


def load_compatible_evidence():
    if COMPARISON_RUN_OVERRIDE is not None:
        candidates = [configured_path(COMPARISON_RUN_OVERRIDE)]
    else:
        candidates = sorted(
            (ROOT / "data/processed/business_model_comparison").glob("*/manifest.json"),
            reverse=True,
        )
    failures = []
    for path in candidates:
        try:
            comparison = load_run(path.parent if path.name == "manifest.json" else path)
            parent = comparison.manifest.get("parent_run")
            if not parent:
                raise ValueError("Run 05 thiếu parent_run từ notebook 03")
            parent_path = configured_path(parent["directory"])
            if POC_RUN_OVERRIDE is not None and parent_path != configured_path(POC_RUN_OVERRIDE):
                raise ValueError("Run 05 không bắt nguồn từ POC_RUN_OVERRIDE")
            poc = load_run(parent_path)
            if sha256(parent_path / "manifest.json") != parent["manifest_sha256"]:
                raise ValueError("Hash manifest cha 03 đã thay đổi")
            for key in [
                "grain",
                "config",
                "protocol",
                "source_signature",
                "catalog_signature",
                "panel_signatures",
            ]:
                if poc.manifest[key] != comparison.manifest[key]:
                    raise ValueError(f"Run 03/05 không cùng {key}")
            if not set(MODEL_NAMES).issubset(comparison.manifest["model_names"]):
                raise ValueError("Run 05 chưa chứa đủ sáu model")
            if EXPECTED_SOURCE_CATEGORY is not None:
                if comparison.manifest["config"]["source_category"] != EXPECTED_SOURCE_CATEGORY:
                    raise ValueError("Run có source category khác cấu hình mong muốn")
            return poc, comparison, failures
        except (OSError, ValueError, KeyError, RuntimeError) as error:
            if COMPARISON_RUN_OVERRIDE is not None:
                raise
            failures.append({"run": path.name, "reason": str(error)[:240]})
    raise RuntimeError(
        "Chưa có cặp run 03/05 COMPLETE tương thích. Chạy lại 03 rồi 05. " + str(failures[:3])
    )


poc, comparison, skipped_runs = load_compatible_evidence()
metrics = comparison.metrics.copy()
config = comparison.manifest["config"]
models = load_models(comparison.directory)
hgb = models["hgb"].estimator
parameters = hgb.get_params(deep=False)
assert {key: str(value) for key, value in parameters.items()} == {
    key: str(value) for key, value in models["hgb"].parameters.items()
}
production_before = file_hashes(PRODUCTION_BUNDLE_DIR)
RUN_DIR = OUTPUT_ROOT / ("run_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ"))
(RUN_DIR / "figures").mkdir(parents=True)
(SOURCE_COPY := RUN_DIR / "notebook_source.ipynb").write_bytes(SOURCE.read_bytes())

matched = metrics.loc[metrics["population"].eq("accepted_weather")].copy()
validation = matched.loc[matched["stage"].eq("validation")].set_index("model")
assert not validation.index.duplicated().any()
assert comparison.manifest["selection_frozen_before_test"]
for stage in ["validation", "test"]:
    same_rows = matched.loc[matched["stage"].eq(stage)]
    for column in ["businesses", "observations", "recorded_checkins"]:
        assert same_rows[column].nunique() == 1, f"Mẫu chấm khác nhau: {stage}/{column}"
selection_order = validation.reindex(comparison.manifest["model_names"]).sort_values(
    "poisson_deviance", kind="stable"
)
selected_model = comparison.manifest["selected_model"]
assert selected_model == selection_order.index[0]
best_calendar = (
    validation.loc[CALENDAR_MODELS].sort_values("poisson_deviance", kind="stable").index[0]
)
best_weather = (
    validation.loc[WEATHER_MODELS].sort_values("poisson_deviance", kind="stable").index[0]
)
print(
    "Ngách:",
    config["city"],
    config["state"],
    "→",
    config["category"],
    "→",
    config["source_category"],
)
print("03:", poc.directory.relative_to(ROOT))
print("05:", comparison.directory.relative_to(ROOT))
if skipped_runs:
    print(
        f"Đã bỏ qua {len(skipped_runs)} run cũ/không tương thích; không ghép metrics từ các run đó."
    )

In [ ]:
display(
    Markdown(
        f"**Kết quả validation của lần chạy này:** {MODEL_NAMES[selected_model]} được chọn; "
        f"Poisson deviance **{validation.loc[selected_model, 'poisson_deviance']:.5f}**. "
        f"Calendar tốt nhất: **{MODEL_NAMES[best_calendar]}**; "
        f"nhánh weather tốt nhất: **{MODEL_NAMES[best_weather]}**. "
        "Các bảng dưới dùng đúng cohort/panel đã xác minh và giữ nguyên lựa chọn trước test."
    )
)

## Context & Methods

**Câu hỏi:** trong cùng một ngách `City/State → Category → Source Category`, model nào
ước lượng activity của **các business đã có lịch sử** tốt hơn calendar baseline?
Đây không phải bài toán quyết định nơi mở một listing mới.

| Giai đoạn | Dữ liệu fit | Dữ liệu chấm | Vai trò |
| --- | --- | --- | --- |
| Validation | 2017–2019 | 2020 | Chọn model bằng Poisson deviance; MAE bổ sung |
| Test | 2018–2020 | 2021 | Đánh giá model đã chọn, không đổi lựa chọn |
| Final | 2019–2021 | Không có nhãn tương lai | Fit model phục vụ kịch bản |

### Key Assumptions

- Cohort của mỗi giai đoạn chỉ dựa vào cửa sổ training, sau đó giữ cố định khi chấm.
  Không lọc bằng `last_checkin` tương lai; không lấy trạng thái mở cửa hiện tại làm chứng cứ năm cũ.
- Một dòng là `business_id × calendar day`; mục tiêu là **số đếm**, không chuẩn hóa theo số business.
- Category cha và Source Category dùng đúng nhãn nguồn và đúng city/state. Một listing có nhiều tag
  có thể thuộc nhiều ngách; không cộng các ngách chồng lặp thành tổng không trùng lặp.
- Ngày nằm trong coverage đã xác nhận mà không có event là **0**. Ngoài coverage hoặc đọc dữ liệu lỗi
  là **không biết**; không điền 0 cho các khoảng đó.
- Calendar fit trên **toàn bộ ngày covered và parent-category pool**, như calendar reference;
  các model weather fit trên training rows đủ accepted weather. Chấm cùng dòng đánh giá
  giúp so cách dự đoán, nhưng chênh lệch điểm **không cô lập lợi ích riêng của weather**
  vì dữ liệu training cũng khác nhau. Cần một ablation giữ nguyên training để kết luận điều đó.
- Weather model được chấm trên cùng các business-day có weather được chấp nhận. Hai calendar baseline
  còn có bảng riêng cho toàn bộ ngày covered; không so trực tiếp hai mẫu chấm khác nhau.
- Weather quan sát ngày đích chỉ có ý nghĩa dự đoán hồi cứu có điều kiện. Kịch bản 2026 dùng weather
  nhập tay, chưa phải dự báo thời tiết hoạt động tại thời điểm lập kế hoạch.

### Xác minh nguồn và quan hệ cha–con

Loader kiểm tra artifact, code, lockfile, dữ liệu nguồn và hash manifest cha.
Hai run phải có cùng protocol, source, cohort và panel. Bảng chính chỉ dùng metrics của
run 05 đã đánh giá đủ model; không ghép Poisson của một protocol khác vào kết quả.

In [ ]:
lineage = pd.DataFrame(
    [
        {
            "Bằng chứng": "03 — POC business",
            "Run": poc.directory.name,
            "Manifest SHA256": sha256(poc.directory / "manifest.json"),
        },
        {
            "Bằng chứng": "05 — So sánh đủ model",
            "Run": comparison.directory.name,
            "Manifest SHA256": sha256(comparison.directory / "manifest.json"),
        },
    ]
)
display(lineage)
display(
    pd.DataFrame([config]).rename(
        columns={
            "city": "City",
            "state": "State",
            "category": "Category",
            "source_category": "Source category",
            "max_businesses": "Giới hạn cohort",
            "seed": "Seed",
        }
    )
)
metrics.to_csv(RUN_DIR / "metrics_verified.csv", index=False)
lineage.to_csv(RUN_DIR / "lineage.csv", index=False)

## Data — Business nào và ngày nào được dùng?

Tên business và tọa độ giúp đọc kết quả, không biến snapshot profile thành dữ liệu lịch sử.
`stars`, `review_count`, `is_open` hiện tại không phải predictor của các model này.
Business identity được học từ cohort training, vì vậy model có thể nhớ mức activity riêng
của từng listing. Điều đó hữu ích để dự đoán listing đã biết nhưng không chứng minh khả năng
ước lượng cho một quán vừa mở.

### Mẫu weather bao phủ bao nhiêu lịch sử?

Tỷ lệ dưới đây là **business-day đánh giá đủ accepted weather / toàn bộ business-day covered**,
không phải tỷ lệ listing có dữ liệu và không phải completeness trung bình đều của các trạm.
Nhiều business có thể dùng chung một trạm; thiếu báo cáo trạm có thể loại đồng thời nhiều listing
và nhiều ngày của cùng mùa. Không giả định các ngày thiếu weather được chọn ngẫu nhiên.
**Kết quả matched-weather chỉ đại diện các dòng còn lại, không phải activity đầy đủ cả năm.**

In [ ]:
weather_coverage = pd.DataFrame(
    [
        {
            "stage": stage,
            "covered_business_days": counts["assessment_days"],
            "accepted_weather_business_days": counts["accepted_assessment_weather_days"],
            "weather_coverage_pct": 100
            * counts["accepted_assessment_weather_days"]
            / counts["assessment_days"],
        }
        for stage, counts in comparison.manifest["stages"].items()
        if stage in ["validation", "test"]
    ]
)
display(
    weather_coverage.rename(
        columns={
            "stage": "Giai đoạn",
            "covered_business_days": "Business-day covered",
            "accepted_weather_business_days": "Business-day đủ weather",
            "weather_coverage_pct": "Weather coverage (%)",
        }
    ).round(2)
)
weather_coverage.to_csv(RUN_DIR / "weather_coverage.csv", index=False)

In [ ]:
data_scope = metrics.loc[
    metrics["model"].isin(["month_weekday_shrunk", "hgb"]),
    ["stage", "model", "population", "businesses", "observations", "recorded_checkins"],
]
display(data_scope)
assert comparison.manifest["grain"] == "business_id_calendar_day"
assert not comparison.predictions.duplicated(
    ["stage", "model", "population", "business_id", "day"]
).any()

### HGB học như thế nào?

HGB (*Histogram-based Gradient Boosting*) chia đầu vào thành các khoảng rồi học nhiều cây nhỏ
nối tiếp nhau. Mỗi vòng sửa một phần lỗi còn lại. Với Poisson loss, model ước lượng số đếm
không âm; **Poisson regression** là một kiến trúc khác cũng dùng mục tiêu Poisson.
Sơ đồ là minh họa, không phải cây được trích từ estimator.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.0))
ax.set(xlim=(0, 13), ylim=(0, 3))
ax.axis("off")
for x, title, body, color in [
    (0.1, "Đầu vào", "Business đã biết\nTháng · thứ · weather", "#e5eef8"),
    (3.5, "Histogram", "Gộp giá trị thành khoảng\nđể tìm điểm chia", "#f6edd5"),
    (6.9, "Nhiều cây nhỏ", "Mỗi cây sửa một phần\nlỗi còn lại", "#e2efe8"),
    (10.3, "Đầu ra", "Recorded check-ins\nước lượng / ngày", "#e5eef8"),
]:
    ax.add_patch(
        FancyBboxPatch(
            (x, 0.7), 2.5, 1.6, boxstyle="round,pad=0.1", facecolor=color, edgecolor="#bbc7ca"
        )
    )
    ax.text(x + 1.25, 1.9, title, ha="center", weight="bold")
    ax.text(x + 1.25, 1.35, body, ha="center", va="center", fontsize=9)
    if x < 10:
        ax.annotate(
            "",
            xy=(x + 3.25, 1.5),
            xytext=(x + 2.65, 1.5),
            arrowprops={"arrowstyle": "->", "color": "#67767b"},
        )
fig.tight_layout()
fig.savefig(RUN_DIR / "figures/hgb_explained.png", dpi=150)
plt.show()

### Tham số của HGB đã lưu

Các giá trị dưới đây lấy trực tiếp từ estimator **final của run 05**, không phải số minh họa
hay cấu hình được giả định từ notebook cũ. Một cấu hình cố định không phải bằng chứng đã tuning tối ưu.

In [ ]:
parameter_meanings = {
    "loss": "Mục tiêu chấm lỗi khi học số đếm.",
    "max_iter": "Số vòng boosting tối đa.",
    "learning_rate": "Mức sửa mỗi vòng; tương tác với số vòng.",
    "max_leaf_nodes": "Độ chi tiết tối đa của mỗi cây.",
    "min_samples_leaf": "Số dòng tối thiểu ở một lá.",
    "l2_regularization": "Phạt độ lớn của bước sửa để hạn chế quá khớp.",
    "early_stopping": "Có tự dừng sớm theo dữ liệu nội bộ hay không.",
    "random_state": "Seed giúp tái lập cấu hình ngẫu nhiên.",
}
parameter_table = pd.DataFrame(
    [
        {
            "Tham số": key,
            "Giá trị thực tế": str(value),
            "Ý nghĩa": parameter_meanings.get(key, "Tham số estimator đã lưu."),
        }
        for key, value in parameters.items()
    ]
)
display(parameter_table)
display(
    pd.DataFrame([{"Số vòng thực chạy": hgb.n_iter_, "Số feature model nhận": hgb.n_features_in_}])
)
parameter_table.to_csv(RUN_DIR / "hgb_parameters.csv", index=False)

## Results — So sánh trên cùng business-day

Mỗi model bên dưới dùng đúng các dòng có weather được chấp nhận của cùng giai đoạn.
Calendar **không dùng weather** nhưng vẫn được chấm trên các dòng đó để so sánh công bằng.
Poisson deviance và MAE đều thấp hơn là tốt hơn. Mỗi business-day là một đơn vị chấm;
không dùng mẫu số 100 business, và business hoạt động mạnh có thể chi phối sai số gộp.

**2020 đã được dùng chọn model.** Xem 2021 để nhận diện suy giảm hoặc khác biệt giữa business;
không quay lại chọn lại model từ test. Những năm này đã được phân tích trong nghiên cứu trước,
nên đây là backtest thăm dò, chưa phải tập kiểm chứng chưa từng được xem.

In [ ]:
score_table = matched.pivot(index="model", columns="stage", values=["poisson_deviance", "mae"])
score_table = score_table.reindex(MODEL_NAMES)
score_table.columns = [f"{stage} · {measure}" for measure, stage in score_table.columns]
score_table.insert(0, "Model", [MODEL_NAMES[name] for name in score_table.index])
display(score_table.round(5))
score_table.to_csv(RUN_DIR / "model_comparison.csv", index_label="model")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, stage, label in zip(
    axes,
    ["validation", "test"],
    ["Validation 2020 — dùng chọn model", "Test 2021 — chỉ đánh giá"],
    strict=True,
):
    values = matched.loc[matched["stage"].eq(stage)].set_index("model").reindex(MODEL_NAMES)
    ax.barh(
        [MODEL_NAMES[name] for name in values.index],
        values["poisson_deviance"],
        color=["#d6ad52" if name in CALENDAR_MODELS else "#317b63" for name in values.index],
    )
    ax.set_title(label)
    ax.set_xlabel("Poisson deviance / business-day · thấp hơn tốt hơn")
axes[0].invert_yaxis()
fig.tight_layout()
fig.savefig(RUN_DIR / "figures/model_comparison.png", dpi=150)
plt.show()

### Calendar trên toàn bộ ngày covered

Bảng này bao gồm cả ngày thiếu weather và giữ nguyên calendar cohort theo training.
Đây là câu hỏi về khả năng bao phủ của calendar reference, không phải fallback tự động trong app. Không dùng độ chênh giữa bảng này và
bảng accepted-weather để kết luận model weather tốt hơn hoặc kém hơn.

In [ ]:
calendar_all = metrics.loc[
    metrics["population"].eq("all_covered"),
    [
        "stage",
        "model",
        "businesses",
        "observations",
        "poisson_deviance",
        "mae",
        "recorded_checkins",
        "predicted_checkins",
    ],
].copy()
display(calendar_all.round(5))
calendar_all.to_csv(RUN_DIR / "calendar_all_covered.csv", index=False)

### Sai số có đồng đều giữa các business không?

Hiển thị một số business có nhiều check-in được ghi nhận trong test để kiểm tra ảnh hưởng
của listing hoạt động mạnh. Việc chọn các dòng để đọc không thay đổi model đã chọn ở validation.

In [ ]:
business_scores = comparison.business_metrics.loc[
    comparison.business_metrics["stage"].eq("test")
    & comparison.business_metrics["population"].eq("accepted_weather")
].copy()
active_ids = (
    business_scores.loc[business_scores["model"].eq("hgb")]
    .sort_values("recorded_checkins", ascending=False)["business_id"]
    .head(TOP_N)
)
display(
    business_scores.loc[business_scores["business_id"].isin(active_ids)]
    .sort_values(["business_id", "model"])
    .round(5)
)

### Trong cùng ngách: business nào đứng đầu ở một kịch bản?

Bảng dùng **HGB để minh họa**, không mặc định đó là model thắng. Mọi business nhận cùng
weather nhập tay và cùng ngày; calendar được ghép theo đúng `business_id` để làm mốc.
Thứ hạng chỉ có nghĩa trong cohort final đã được huấn luyện và cấu hình này.
Tọa độ là vị trí listing sẵn có, không phải các địa điểm mới được đề xuất mở quán.

In [ ]:
assert MANUAL_WEATHER["tmin_c"] <= MANUAL_WEATHER["tmax_c"]
assert MANUAL_WEATHER["precipitation_mm"] >= 0
scenario_ranking = recommend(
    comparison.directory, DEMO_DAY, model="hgb", weather=MANUAL_WEATHER, top_n=TOP_N
)
assert not scenario_ranking.empty
assert scenario_ranking["business_id"].is_unique
calendar_ranking = recommend(comparison.directory, DEMO_DAY, model=best_calendar, top_n=100_000)
scenario_table = scenario_ranking.merge(
    calendar_ranking[["business_id", "predicted_checkins"]].rename(
        columns={"predicted_checkins": "calendar_checkins"}
    ),
    on="business_id",
    how="left",
    validate="one_to_one",
)
assert scenario_table["calendar_checkins"].notna().all()
display(
    scenario_table[
        [
            "business_id",
            "name",
            "source_categories",
            "predicted_checkins",
            "calendar_checkins",
            "latitude",
            "longitude",
        ]
    ].round(3)
)
scenario_table.to_csv(RUN_DIR / "business_scenario_ranking.csv", index=False)

### So hai business bằng Recorded check-ins theo tháng

Mặc định chọn hai business từ bảng kịch bản có mặt trong cohort test đã lưu.
Nếu ngách chỉ có một business đủ điều kiện, biểu đồ chỉ có một đường.
Các đường biểu diễn tổng số đếm **đầy đủ các ngày covered trong cùng tháng** của test 2021.
Biểu đồ không phụ thuộc ngày đủ weather, không phải rate và không tự coi lịch sử ngoài coverage là 0.

In [ ]:
test_history_ids = set(
    comparison.predictions.loc[
        comparison.predictions["stage"].eq("test")
        & comparison.predictions["model"].eq("month_weekday_shrunk")
        & comparison.predictions["population"].eq("all_covered"),
        "business_id",
    ]
)
chosen_ids = (
    COMPARE_BUSINESS_IDS
    or scenario_table.loc[scenario_table["business_id"].isin(test_history_ids), "business_id"]
    .head(2)
    .tolist()
)
if not set(chosen_ids).issubset(test_history_ids):
    raise ValueError("Business không thuộc cohort test; chọn ID có lịch sử test đã lưu")
if not 1 <= len(chosen_ids) <= 2 or len(set(chosen_ids)) != len(chosen_ids):
    raise ValueError("Chọn một hoặc hai business_id khác nhau")
known_businesses = calendar_ranking.set_index("business_id")
if not set(chosen_ids).issubset(known_businesses.index):
    raise ValueError("Business ngoài cohort final đã fit; không suy diễn sang listing mới")
profiles = known_businesses.loc[chosen_ids, ["name", "source_categories", "latitude", "longitude"]]
display(profiles)
recorded = comparison.predictions.loc[
    comparison.predictions["stage"].eq("test")
    & comparison.predictions["model"].eq("month_weekday_shrunk")
    & comparison.predictions["population"].eq("all_covered")
    & comparison.predictions["business_id"].isin(chosen_ids),
    ["business_id", "day", "recorded_checkins"],
].copy()
recorded["month"] = pd.to_datetime(recorded["day"]).dt.to_period("M").dt.to_timestamp()
assert set(chosen_ids).issubset(set(recorded["business_id"]))
monthly = recorded.groupby(["month", "business_id"])["recorded_checkins"].sum().unstack()
fig, ax = plt.subplots(figsize=(11, 3.8))
for business_id in chosen_ids:
    if business_id in monthly:
        ax.plot(
            monthly.index,
            monthly[business_id],
            marker="o",
            label=f"{known_businesses.loc[business_id, 'name']} · {business_id[:8]}",
        )
ax.set(title="Business trong cùng ngách · test 2021", ylabel="Recorded check-ins / tháng")
ax.legend(fontsize=9)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(RUN_DIR / "figures/two_businesses_recorded.png", dpi=150)
plt.show()
monthly.to_csv(RUN_DIR / "two_businesses_monthly.csv")

### Giữ nguyên một business, chỉ đổi weather năm 2026

Ba dòng sau giữ nguyên business identity, ngày và calendar. Weather là giả định chung,
không phải nguồn khí tượng dự báo năm 2026. Nếu HGB đổi kết quả, đó là quan hệ model đã học;
không diễn giải thành tác động nhân quả của mưa hay nhiệt độ.

In [ ]:
demo_business_id = chosen_ids[0]
scenario_rows = []
for label, weather in WEATHER_EXAMPLES.items():
    row = (
        recommend(comparison.directory, DEMO_DAY, model="hgb", weather=weather, top_n=100_000)
        .set_index("business_id")
        .loc[demo_business_id]
    )
    scenario_rows.append(
        {
            "Kịch bản": label,
            **weather,
            "HGB check-ins": row["predicted_checkins"],
            "Calendar check-ins": known_businesses.loc[demo_business_id, "predicted_checkins"],
        }
    )
weather_scenarios = pd.DataFrame(scenario_rows)
display(
    Markdown(
        f"**Business giữ cố định:** {known_businesses.loc[demo_business_id, 'name']} "
        f"(`{demo_business_id}`) · **ngày giả định:** {DEMO_DAY}."
    )
)
display(weather_scenarios.round(3))
fig, ax = plt.subplots(figsize=(10, 3.8))
weather_scenarios.set_index("Kịch bản")[["HGB check-ins", "Calendar check-ins"]].plot.bar(
    ax=ax, color=["#317b63", "#d6ad52"], rot=0
)
ax.set(ylabel="Projected recorded check-ins / business / ngày", xlabel="")
fig.tight_layout()
fig.savefig(RUN_DIR / "figures/weather_scenarios.png", dpi=150)
plt.show()
weather_scenarios.to_csv(RUN_DIR / "weather_scenarios.csv", index=False)

## Takeaways

Kết luận dưới đây tính từ bảng validation đã xác minh và chỉ đọc test để mô tả.
Giới hạn cần giữ khi sử dụng: cohort hữu hạn, identity đã biết, weather ngày đích,
phụ thuộc coverage và thay đổi hành vi giữa các năm. Năm 2026 không có nhãn để chấm accuracy.
Recorded check-ins là proxy activity, không phải khách hàng duy nhất, doanh thu,
quy mô thị trường hay xác suất thành công của một quán mới.

Notebook lưu bảng, hình, cấu hình và hash cha trong thư mục nghiên cứu riêng.
Nó không publish artifact serving. `models/business_forecast` là calendar reference được giữ riêng;
app chọn HGB cho khả năng dùng weather scenario và báo unavailable nếu thiếu hỗ trợ, không tự fallback calendar.

In [ ]:
heldout = matched.loc[matched["stage"].eq("test")].set_index("model")
weather_validation = float(validation.loc[best_weather, "poisson_deviance"])
calendar_validation = float(validation.loc[best_calendar, "poisson_deviance"])
relative_change = (
    (weather_validation / calendar_validation - 1) * 100 if calendar_validation else None
)
comparison_phrase = (
    f"cao hơn {relative_change:.2f}%"
    if relative_change is not None and relative_change >= 0
    else f"thấp hơn {-relative_change:.2f}%"
    if relative_change is not None
    else "không tính tỷ lệ vì calendar deviance bằng 0"
)
selected_validation = float(validation.loc[selected_model, "poisson_deviance"])
selected_test = float(heldout.loc[selected_model, "poisson_deviance"])
coverage_lookup = weather_coverage.set_index("stage")["weather_coverage_pct"]
summary = (
    f"Trên ngách {config['source_category']} tại {config['city']}, {config['state']}, "
    f"run 05 chọn **{MODEL_NAMES[selected_model]}** từ validation 2020. "
    f"Model này có deviance validation **{selected_validation:.5f}** "
    f"và test 2021 **{selected_test:.5f}** trên cùng tập ngày đủ weather. "
    f"Nhánh weather tốt nhất theo validation là **{MODEL_NAMES[best_weather]}**; "
    f"deviance {comparison_phrase} so với calendar tốt nhất trên đúng các dòng đó. "
    "Đây là kết quả của cohort/config đã lưu, không chứng minh weather gây ra check-in "
    "hoặc model thắng ở ngách khác. Training calendar và weather khác tập dòng, nên chênh lệch "
    "điểm chưa cô lập lợi ích riêng của weather. HGB được giải thích để hiểu mô hình; "
    "notebook không "
    "ghi đè artifact phục vụ ứng dụng. HGB serving hỗ trợ weather scenario "
    "theo chính sách sản phẩm; "
    "điều đó không có nghĩa HGB thắng accuracy. "
    f"Weather coverage của validation là {coverage_lookup.loc['validation']:.2f}% "
    f"và test là {coverage_lookup.loc['test']:.2f}% business-day covered; "
    "matched-weather metrics không đại diện activity đầy đủ cả năm."
)
display(Markdown(summary))
(RUN_DIR / "summary.md").write_text(summary + "\n", encoding="utf-8")
assert production_before == file_hashes(PRODUCTION_BUNDLE_DIR), "Không được sửa bundle ứng dụng"
manifest = {
    "status": "complete",
    "schema_version": 1,
    "grain": "business_id_calendar_day",
    "created_at_utc": datetime.now(UTC).isoformat(),
    "purpose": "Vietnamese verified individual-business HGB analytical report; no retraining",
    "config": config,
    "protocol": comparison.manifest["protocol"],
    "selected_model": selected_model,
    "best_validation_weather_model": best_weather,
    "parents": [
        {
            "directory": str(run.directory.resolve()),
            "manifest_sha256": sha256(run.directory / "manifest.json"),
        }
        for run in [poc, comparison]
    ],
    "source_signature": comparison.manifest["source_signature"],
    "catalog_signature": comparison.manifest["catalog_signature"],
    "panel_signatures": comparison.manifest["panel_signatures"],
    "notebook_source_sha256": sha256(SOURCE_COPY),
    "lockfile_sha256": sha256(ROOT / "uv.lock"),
    "model_parameters": {key: str(value) for key, value in parameters.items()},
    "production_bundle_unchanged_sha256": production_before,
    "demo": {
        "business_id": demo_business_id,
        "day": DEMO_DAY.isoformat(),
        "manual_weather": MANUAL_WEATHER,
        "interpretation": "conditional_existing_business_only",
    },
    "output_sha256": file_hashes(RUN_DIR),
}
(RUN_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Đã lưu:", RUN_DIR.relative_to(ROOT))

### Đọc tiếp

- [03 — POC ở cấp business](03_location_forecast_poc.ipynb)
- [05 — Poisson, HGB, GAM và CatBoost cùng protocol](05_forecast_gam_catboost_comparison.ipynb)
- [07 — Giải thích HGB cho người mới](07_hgb_for_beginners.ipynb)
- [Workflow của ứng dụng](../docs/business-location-workspace.md)
- [HGB trong scikit-learn](https://scikit-learn.org/stable/modules/ensemble.html#histogram-based-gradient-boosting)